[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/magnetotransport/magnetotransport.ipynb)

# Galvanomagnetic and Thermomagnetic Effects

This example computes the coupled transport of electric charge and heat by electrons and holes across a Hall bar and a Corbino ring in a perpendicular magnetic field as the carrier balance sweeps from pure electrons through compensation to pure holes. Each carrier is modelled as holding an electric charge and a thermal entropy, so an electrochemical potential gradient pushes opposite charges in opposite directions while a temperature gradient pushes both species toward the cooler contact, and collisions balance that push into a steady Drude drift. A perpendicular magnetic field deflects each drifting species sideways through its Hall angle, turning opposite charges opposite ways. Along an insulated Hall bar, a single carrier species builds up a transverse Hall voltage that cancels the magnetic deflection and leaves the longitudinal resistance unchanged, whereas under open-circuit heating its Seebeck voltage holds the carriers at rest. At carrier compensation, as in a semimetal such as bismuth, the opposite transverse charge currents cancel so no Hall voltage forms, magnetoresistance rises with the field, and both species carry entropy to the same edge in what is called the Ettingshausen effect; under a temperature gradient the magnetic field parts electrons and holes to opposite edges, producing a transverse Nernst voltage that peaks where the Hall voltage vanishes. Around a Corbino ring, closed rim contacts prevent transverse charge accumulation so carriers spiral at every balance and radial heating drives a circulating azimuthal current. In a magnetized conductor with zero external magnetic field, spin-dependent skew scattering deflects spin-up and spin-down electrons through opposite angles, producing an anomalous Hall voltage proportional to magnetization and, when unmagnetized, a pure transverse spin current called the spin Hall effect.

In planar projective spacetime with degenerate point weight `w` joined with a null entropy direction `s`, the electrochemical potential and temperature combine into one `Potential` 1-form whose negative wedge with the spatial gradient forms the thermodynamic `drive`. The term ***parcel*** here means an intrinsic point in the space of conserved quantities holding a carrier's charge along time and its entropy, `valence * (time + entropy_per_charge * entropy)`. Commuting the magnetic bivector and the drive with a moving carrier, `(magnetic + drive).commutator(parcel + valence * velocity)`, unites the electrostatic push, the thermal push, and the Lorentz turning while spending zero magnetic power, so Drude's steady drift inverts a single scaled spatial rotor `1 - valence * mobility * magnetic`. Joining each species' velocity with its parcel via `velocity.regressive(parcel)` forms one charge-and-entropy flux whose pairings with `xt`, `yt` and `xs`, `ys` show opposite carrier turns cancelling in charge at compensation while adding in entropy, and reversing the magnetic field reverses each rotor to map the linear transport operator to its `.adjugate()`, expressing Onsager–Casimir reciprocity while keeping the Lorentz-invariant dissipation `drive & flux` positive. A [companion notebook](magnetotransport_one_place.ipynb) evaluates the five bulk transport coefficients at a single interior point without a topology.

In the matrix notation of coupled two-band transport, solving $\mathbf v_i = \mu_i(\zeta_i e\,\mathbf E - s_i\nabla T + \zeta_i e\,\mathbf v_i \times \mathbf B)$ for bands $i \in \{e, h\}$ and assembling separate charge and heat currents $\mathbf J_e = \sum_i n_i \zeta_i e\,\mathbf v_i$ and $\mathbf J_Q = T\sum_i n_i s_i \mathbf v_i - \kappa_L \nabla T$ introduces three $2\times 2$ block tensors $\boldsymbol\sigma(\mathbf B)$, $\boldsymbol\alpha(\mathbf B)$, and $\bar{\boldsymbol\kappa}(\mathbf B)$ with explicit carrier-sign factors $\zeta_i = \pm 1$ and skew off-diagonal entries $\pm \zeta_i \mu_i B / (1 + \mu_i^2 B^2)$. Enforcing open-circuit thermal boundary conditions and Onsager–Casimir reciprocity then requires the block-matrix inversions $\boldsymbol\rho = \boldsymbol\sigma^{-1}$, $\mathbf S = \boldsymbol\rho\boldsymbol\alpha$, and $\boldsymbol\kappa = \bar{\boldsymbol\kappa} - T\boldsymbol\alpha\boldsymbol\rho\boldsymbol\alpha^{\mathsf T}$ alongside the three separate transpose identities $\boldsymbol\sigma^{\mathsf T}(\mathbf B) = \boldsymbol\sigma(-\mathbf B)$, $\bar{\boldsymbol\kappa}^{\mathsf T}(\mathbf B) = \bar{\boldsymbol\kappa}(-\mathbf B)$, and $T\boldsymbol\alpha^{\mathsf T}(\mathbf B) = \boldsymbol\Pi(-\mathbf B)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.thermodynamics.magnetotransport import render, scenarios
from examples.util.animation import show_animation

# Planar projective spacetime with degenerate weight w, joined with a null direction s for entropy.
ga = Algebra("x+y+t-w0") * Algebra("s0")
space = Algebra("x+y+")
quantities = Algebra("t-s0")
mv = NumpyContext(ga).multivector

Scalar = ga.gatype.scalar()                                                   # Scalar+
Antiscalar = ga.gatype.antiscalar()                                           # Antiscalar-
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Parcel = Plane.derive.within(quantities).derive.dual                          # Antivector-
Spatial = Plane.derive.within(space).derive.dual                              # Antivector-
Upright = Plane.derive.within(space)                                          # Vector+
Potential = Plane.derive.within(quantities)                                   # Vector+
Drive = ga.gatype.bivector().derive.touching(space, quantities)               # Bivector+
Magnetic = ga.gatype.bivector().derive.within(space)                          # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
DriftMap = ga.gatype((Spatial, Drive))                                        # Spatial- <- Drive+
Response = ga.gatype((Flux, Drive))                                           # Flux- <- Drive+

x_direction = mv.x.dual()                                                     # [] Spatial-
y_direction = mv.y.dual()                                                     # [] Spatial-
time = -mv.t.dual()                                                           # [] Parcel-
entropy = mv.s.dual()                                                         # [] Parcel-
origin = mv.w.dual()                                                          # [] Point-
spin_plane = mv.xy                                                            # [] Magnetic+

## Carrier drift and the Hall rotor

Each carrier species drifts at the velocity where collisional drag, `velocity / mobility`, balances the combined force of the drive and the magnetic field. Commuting `magnetic + drive` with the moving carrier `parcel + valence * velocity` yields the electrostatic push on charge, the thermal push on entropy, and the Lorentz turning across space in one operation while contributing zero magnetic power along `t`. Because the commutator of a spatial magnetic bivector with a spatial velocity reduces to their geometric product, solving `velocity == mobility * (rest_push + valence * magnetic * velocity)` inverts `1 - valence * mobility * magnetic`. In two dimensions a scalar plus a spatial bivector is a rotor scaled by the secant of its turning angle, so its inverse rotates the rest push through the species' Hall angle and foreshortens it by the cosine, turning electrons and holes opposite ways.

In [ ]:
# Electrons and holes carry opposite charges, opposite entropy per unit charge, unit mobility, and zero spin skew.
valences = np.array([-1, 1])                                                  # [2]
entropies_per_charge = np.array([-1.0, 1.0])                                  # [2]
mobilities = np.array([1.0, 1.0])                                             # [2]
skews = np.zeros(2)                                                           # [2]
strength = 2.0

In [ ]:
def drifts(magnetic: Magnetic, valences: np.ndarray, entropies_per_charge: np.ndarray, mobilities: np.ndarray,
           skews: np.ndarray) -> DriftMap:
    """Drude steady balance between collision drag, Lorentz turning, and spin skew scattering maps a drive to each carrier species' drift velocity (`[..., 2] Spatial- <- Drive+`)."""
    parcels = valences * (time + entropies_per_charge * entropy)              # [2] Parcel-
    # Commuting the drive and magnetic field with a moving parcel separates into the rest push on the parcel plus magnetic turning of its spatial drift.
    rest_push = Drive.commutator(parcels)                                     # [2] Spatial- <- Drive+
    # Magnetic turning and spin skew scattering combine with collision drag into one scaled spatial rotor per carrier species.
    hall_rotors = 1 - valences * mobilities * magnetic[..., None] - skews * spin_plane   # [..., 2] Rotor+
    return hall_rotors.inverse() * (mobilities * rest_push)


parcels = valences * (time + entropies_per_charge * entropy)                  # [2] Parcel-
magnetic = strength * mv.xy                                                   # [] Magnetic+
hall_rotors = 1 - valences * mobilities * magnetic                            # [2] Rotor+
carrier_drifts = drifts(magnetic, valences, entropies_per_charge, mobilities, skews)   # [2] Spatial- <- Drive+
# Extracting the logarithm of the normalized rotor along xy gives each species' Hall angle.
hall_angles = hall_rotors.normalized().log().scalar_product(~mv.xy)           # [2] Scalar+

print(f"Hall angles of electrons and holes: {np.degrees(hall_angles.to_array()).round(1)} degrees")

## Charge and entropy in one flux

Joining a species' spatial drift velocity with its parcel via `velocity.regressive(parcel)` forms an antibivector `Flux` that carries charge current and entropy current together: pairing with `xt` and `yt` reads longitudinal and transverse charge flux, while pairing with `xs` and `ys` reads longitudinal and transverse entropy flux. Under a unit longitudinal electric drive, each species' flux turns through its own Hall angle; at carrier compensation, where electrons and holes have equal density, the opposite transverse charge components cancel while the entropy components, which electrons carry opposite to their negative charge, add straight across the conductor.

In [ ]:
carrier_fluxes = carrier_drifts.regressive(parcels)                           # [2] Flux- <- Drive+
unit_drive = mv.xt                                                            # [] Drive+
end_fluxes = carrier_fluxes(unit_drive)                                       # [2] Flux-
render.turns_figure(unit_drive, end_fluxes);

The crimson arrow shows the unit longitudinal electric drive read against `x_direction & time` and `y_direction & time`. In the left panel, pairing the fluxes for charge places the grey pure-electron current and the black pure-hole current at opposite Hall angles on the circle having the crimson arrow as diameter; as carrier balance sweeps along the dotted chord, the purple compensated midpoint aligns purely along the drive with magnitude reduced by the squared cosine of the Hall angle. In the right panel, pairing the same fluxes for entropy flips the electron arrow because electrons carry positive entropy opposite to their negative charge, so the Peltier entropy currents sum at compensation into a purple arrow pointing purely across the drive toward the edge warmed by the Ettingshausen effect.

## Triangulated Hall bar and Corbino ring

The Hall bar spans three widths between end contacts with longitudinal and transverse voltage probes along its lower edge and midline, while the Corbino ring spans inner radius `0.4` to outer radius `1.0` between concentric rim contacts; both triangulations place unit-weight vertices at `origin` plus spatial offsets. The term ***simplex gradient*** here means the spatial covector on each triangle corner that pairs to zero along the opposite edge and rises to one at its own vertex, formed by joining the two opposite corners with `time & entropy`, dividing by the volume join closed at the corner, and projecting onto `Upright`.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Upright]:
    """Joining opposite corners of each triangle gives its area (`Scalar+[F]`) and its three corner simplex gradients (`[3] Upright+[F]`)."""
    corners = topology.incidence[2, 0] * vertices                             # [3] Point-[F]
    facing = topology.off[2, 0] * vertices                                    # [2, 3] Point-[F]
    # Joining the opposite edge with time and entropy forms the hyperplane of constant off-plane potentials, normalized to read one at the corner.
    sides = facing[0] & facing[1]                                             # [3] Antibivector-[F]
    unread = time & entropy                                                   # [] Antibivector-
    gradients = ((sides & unread) / (sides & unread & corners)).cast(Upright)  # [3] Upright+[F]
    double_areas = corners[0] & corners[1] & corners[2] & unread              # Antiscalar-[F]
    return double_areas.norm() / 2, gradients


bar_topology, bar_vertices, left_contacts, right_contacts, plus_probes, minus_probes = scenarios.bar(scenarios.n_rows, scenarios.n_columns)
ring_topology, ring_vertices, inner_contacts, outer_contacts = scenarios.ring(scenarios.n_rings, scenarios.n_sectors)
bar_areas, bar_gradients = triangles(bar_topology, bar_vertices)              # Scalar+[F], [3] Upright+[F]
ring_areas, ring_gradients = triangles(ring_topology, ring_vertices)          # Scalar+[F], [3] Upright+[F]
render.conductors_figure(bar_topology, bar_vertices, (left_contacts, right_contacts), ring_topology,
                       ring_vertices, (inner_contacts, outer_contacts), scenarios.bar_resolution,
                       scenarios.ring_resolution);

## Steady potentials under current and heat drives

Wedging a corner's simplex gradient with a triangle's flux, `gradients ^ flux`, measures the charge and entropy carried out of that vertex's dual cell across the triangle. Pulling the local `3 x 3` triangle coupling back through the corner-vertex incidence `topology.incidence[2, 0]` assembles the global dual-cell outflow operator across all vertices. In linear transport, quadratic Joule and thermal-dissipation entropy sources drop out, so every free dual cell has zero net outflow while insulated borders pass neither charge nor entropy. Solving unit electrochemical and thermal potential drops between the held contacts provides two linear basis solutions: scaling the first enforces a prescribed isothermal current, and subtracting a matching multiple of the first from the second cancels the thermoelectric short-circuit current to leave an open-circuit heat-driven state.

In [ ]:
def outflow(topology: Topology, areas: Scalar, gradients: Upright, response: Response) -> SparseExtensor:
    """Wedging triangle simplex gradients with the driven flux assembles the dual-cell charge and entropy outflow operator (`(Parcel- <- Potential+)[V <- V]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    corner_drives = -(gradients ^ Potential)                                  # [3] (Drive+ <- Potential+)[F]
    triangle_outflow = areas * (gradients[:, None] ^ response(corner_drives)[None, :])   # [3, 3] (Parcel- <- Potential+)[F]
    return (corners[:, None].adjugate() * spdiag(triangle_outflow) * corners[None, :]).sum(axis=0).sum(axis=0)


def steady_potentials(topology: Topology, areas: Scalar, gradients: Upright, response: Response,
                      high_contacts: np.ndarray, low_contacts: np.ndarray, current: float,
                      temperature_difference: float) -> tuple[Potential, Potential]:
    """Zero net charge and entropy outflow on free dual cells determines the isothermal current-driven and open-circuit heat-driven vertex potentials (`Potential+[V]`)."""
    V, E, F = topology.n_elements
    contact_signs = np.zeros(V)
    contact_signs[high_contacts], contact_signs[low_contacts] = 1, -1
    # Solving unit electrochemical and thermal drops between the contacts gives two linear basis states whose combination enforces fixed current or open circuit.
    contact_potentials = mv.scalar(contact_signs[:, None]).field() * stack((mv.t, mv.s)) / 2   # [2] Potential+[V]
    free_reading = topology.reading(np.flatnonzero(contact_signs == 0), 0)    # Scalar+[V_free <- V]
    vertex_outflow = outflow(topology, areas, gradients, response)            # (Parcel- <- Potential+)[V <- V]
    free_outflow = free_reading * vertex_outflow * free_reading.adjugate()    # (Parcel- <- Potential+)[V_free <- V_free]
    basis_potentials = contact_potentials + free_reading.adjugate() * free_outflow.solve(
        -(free_reading * vertex_outflow(contact_potentials)))                 # [2] Potential+[V]
    # Charge leaving the high-contact dual cells gives the net current entering the conductor under each basis drop.
    contact_currents = mv.t & (topology.reading(high_contacts, 0) * vertex_outflow(basis_potentials)).sites.sum()   # [2] Scalar+
    unit_electric, unit_thermal = basis_potentials                            # Potential+[V], Potential+[V]
    electric_current, thermal_current = contact_currents                      # [] Scalar+, [] Scalar+
    return (unit_electric * (current / electric_current),
            (unit_thermal - unit_electric * (thermal_current / electric_current)) * temperature_difference)

## Carrier balance sweep

Sweeping the hole-minus-electron balance from `-1` (electrons only) through `0` (compensation) to `+1` (holes only) at unit total carrier density keeps the zero-field electrical conductivity fixed at one while adding phononic lattice entropy conduction down temperature gradients. At each frame, `steady_states` solves both the isothermal current drive and the open-circuit thermal drive on the Hall bar and the Corbino ring. Running and rendering the current-driven animation takes about four seconds.

In [ ]:
conductivity = 1.5
current, temperature_fall = 1.0, 1.0
n_frames, frame_duration_ms = 75, 40
balances = -np.cos(np.linspace(0, np.pi, n_frames))                           # [n_frames]
densities = np.stack([(1 - balances) / 2, (1 + balances) / 2], axis=-1)      # [n_frames, 2]
magnetic_sweep = np.full(n_frames, strength) * mv.xy                          # [n_frames] Magnetic+

In [ ]:
def response(magnetic: Magnetic, valences: np.ndarray, entropies_per_charge: np.ndarray, mobilities: np.ndarray,
             skews: np.ndarray, densities: np.ndarray, conductivity: float) -> Response:
    """Carrier drift fluxes weighted by `densities` (`[..., 2]`) combine with lattice thermal conduction into the linear transport map (`[...] Flux- <- Drive+`)."""
    parcels = valences * (time + entropies_per_charge * entropy)              # [2] Parcel-
    species_drifts = drifts(magnetic, valences, entropies_per_charge, mobilities, skews)   # [..., 2] Spatial- <- Drive+
    carrier_response = (densities * species_drifts.regressive(parcels)).sum(axis=-1)       # [...] Flux- <- Drive+
    lattice_response = conductivity * Drive.commutator(entropy).regressive(entropy)        # [] Flux- <- Drive+
    return carrier_response + lattice_response


def steady_states(topology: Topology, areas: Scalar, gradients: Upright, high_contacts: np.ndarray,
                  low_contacts: np.ndarray, current: float, temperature_difference: float, magnetic: Magnetic,
                  valences: np.ndarray, entropies_per_charge: np.ndarray, mobilities: np.ndarray, skews: np.ndarray,
                  densities: np.ndarray, conductivity: float
                  ) -> Iterator[tuple[np.ndarray, Potential, Spatial, Potential, Spatial]]:
    """Yields per-frame carrier densities (`[2]`), vertex potentials (`Potential+[V]`), and triangle drift velocities (`[2] Spatial-[F]`) under current and heat drives."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    for frame, frame_densities in enumerate(densities):
        frame_drifts = drifts(magnetic[frame], valences, entropies_per_charge, mobilities, skews)   # [2] Spatial- <- Drive+
        frame_response = response(magnetic[frame], valences, entropies_per_charge, mobilities, skews,
                                  frame_densities, conductivity)              # [] Flux- <- Drive+
        electric_potentials, thermal_potentials = steady_potentials(
            topology, areas, gradients, frame_response, high_contacts, low_contacts, current, temperature_difference)
        potentials = stack((electric_potentials, thermal_potentials))         # [2] Potential+[V]
        triangle_drives = -(gradients ^ (corners * potentials[:, None])).sum(axis=-1)   # [2] Drive+[F]
        yield (frame_densities, electric_potentials, frame_drifts(triangle_drives[0]),
               thermal_potentials, frame_drifts(triangle_drives[1]))


carriers = (valences, entropies_per_charge, mobilities, skews)
frames = list(zip(
    magnetic_sweep,
    steady_states(bar_topology, bar_areas, bar_gradients, left_contacts, right_contacts, current,
                  temperature_fall * scenarios.length, magnetic_sweep, *carriers, densities, conductivity),
    steady_states(ring_topology, ring_areas, ring_gradients, inner_contacts, outer_contacts, current,
                  temperature_fall * (1 - scenarios.inner_radius), magnetic_sweep, *carriers, densities, conductivity)))
bar_states = [bar_state for _, bar_state, _ in frames]
ring_states = [ring_state for _, _, ring_state in frames]
conductors = (bar_topology, bar_vertices, (left_contacts, right_contacts), ring_topology, ring_vertices,
              (inner_contacts, outer_contacts), scenarios.bar_resolution, scenarios.ring_resolution)
show_animation(render.current_driven_animation(*conductors, frames[0], frames), "magnetotransport_charge",
               frame_duration_ms, n_frames=n_frames)

The upright Hall bar on the left receives current at its bottom contact with its lower probe edge on the right, beside the blue magnetic-field disc and the Corbino ring. Grey shading marks equilibrium temperature, shifting toward red where warmer and blue where cooler, overlaid with dark electrochemical equipotential lines, white electron drift streaks, and black hole drift streaks inked in proportion to species share. With pure electrons, equipotentials tilt across the bar at the Hall angle so the Hall field keeps electrons flowing straight up the channel; approaching compensation, the equipotentials square up and crowd together as longitudinal resistance rises, while both species deflect toward the right edge and warm it via the Ettingshausen effect, before tilting the opposite way for pure holes. Around the Corbino ring, electrons spiral inward and holes outward with identical concentric equipotentials and resistance `(1 + strength^2)` times the zero-field value at every balance. Rendering the heat-driven animation on the same states takes about four seconds.

In [ ]:
show_animation(render.heat_driven_animation(*conductors, frames[0], frames), "magnetotransport_heat",
               frame_duration_ms, n_frames=n_frames)

Under open-circuit heating from the warm bottom bar contact and warm inner ring contact, a single carrier species stands still against its own transverse-free Seebeck voltage, leaving equipotentials horizontal across the bar and concentric in the ring. Near compensation the Seebeck voltage vanishes so both species drift together down the temperature gradient; in the Hall bar their opposite magnetic deflections build up longitudinal Nernst equipotentials across the channel, whereas in the Corbino ring, with no lateral boundary to arrest azimuthal flow, electrons and holes spiral opposite ways around the annulus to drive a net circulating current.

## Transport coefficients across carrier balance

Differencing vertex potentials across the longitudinal probe pair on the bar's right edge and the transverse Hall probe pair across its midline extracts all five transport coefficients directly from `bar_states`: longitudinal and Hall resistances per unit current under current drive, together with longitudinal Seebeck thermopower, transverse Nernst signal, and the Righi–Leduc transverse-to-longitudinal temperature drop ratio under open-circuit heat drive.

In [ ]:
electric_potentials = stack([potentials for _, potentials, _, _, _ in bar_states])   # [n_frames] Potential+[V]
thermal_potentials = stack([potentials for _, _, _, potentials, _ in bar_states])    # [n_frames] Potential+[V]
voltmeters = bar_topology.reading(plus_probes, 0) - bar_topology.reading(minus_probes, 0)   # Scalar+[2 <- V]
electric_voltages = (voltmeters * (electric_potentials & time)).batch() / current    # [n_frames, 2] Scalar+
thermal_voltages = (voltmeters * (thermal_potentials & time)).batch()                # [n_frames, 2] Scalar+
probe_temperatures = (voltmeters * (thermal_potentials & entropy)).batch()           # [n_frames, 2] Scalar+
longitudinal_drop = probe_temperatures[:, 0]                                  # [n_frames] Scalar+
resistance, hall_resistance = electric_voltages[:, 0], electric_voltages[:, 1]   # [n_frames] Scalar+, [n_frames] Scalar+
seebeck = -thermal_voltages[:, 0] / longitudinal_drop                         # [n_frames] Scalar+
nernst = -thermal_voltages[:, 1] / longitudinal_drop                          # [n_frames] Scalar+
righi_leduc = probe_temperatures[:, 1] / longitudinal_drop                    # [n_frames] Scalar+
render.readings_figure(balances, resistance, hall_resistance, seebeck, nernst, righi_leduc, render.balance_names);

In the top panel, Hall resistance runs from `-strength` for pure electrons through zero at compensation to `+strength` for pure holes (slightly reduced by end-contact shorting), while longitudinal resistance rises from `1` at either endpoint to a peak at compensation that stays below the isothermal value `1 + strength^2` because the transverse Ettingshausen temperature gradient pushes both species back against their magnetic deflection. In the middle panel, Seebeck thermopower matches each pure species' own entropy per charge at the endpoints, satisfying Kelvin's relation by construction, and crosses zero at compensation where the Nernst signal reaches its maximum. In the bottom panel, the Righi–Leduc temperature ratio vanishes at both pure-carrier endpoints (where carriers stand still) and at compensation (where electrons and holes drift symmetrically straight down the bar), peaking with opposite signs at intermediate balances.

## Magnetic field reversal at compensation

Reversing the magnetic field replaces each species' Hall rotor by `hall_rotors.reverse()` and maps the linear transport operator to its `.adjugate()`, expressing Onsager–Casimir reciprocity as a single typed map identity and making transverse responses odd in the field. Sweeping the field from `+strength` through zero to `-strength` at fixed carrier compensation under open-circuit heating shows that odd symmetry directly. Running and rendering the field-reversal animation takes about five seconds.

In [ ]:
reversal = strength * np.cos(np.linspace(0, np.pi, n_frames)) * mv.xy         # [n_frames] Magnetic+
compensation = np.full_like(densities, 0.5)                                   # [n_frames, 2]

In [ ]:
reversal_frames = zip(
    reversal,
    steady_states(bar_topology, bar_areas, bar_gradients, left_contacts, right_contacts, current,
                  temperature_fall * scenarios.length, reversal, *carriers, compensation, conductivity),
    steady_states(ring_topology, ring_areas, ring_gradients, inner_contacts, outer_contacts, current,
                  temperature_fall * (1 - scenarios.inner_radius), reversal, *carriers, compensation, conductivity))
show_animation(render.heat_driven_animation(*conductors, frames[0], reversal_frames), "magnetotransport_reversed",
               frame_duration_ms, n_frames=n_frames)

As the blue magnetic-field disc shrinks through a point and re-expands with its rim chevron reversed, the outward-drifting electrons and holes in the Corbino ring straighten into radial rays at zero field and then spiral in opposite senses, reversing the circulating azimuthal current. In the Hall bar, the Nernst voltage reverses sign across the channel while keeping equipotential lines parallel to the edges, and near the shorting end contacts the white electron streaks and black hole streaks swap sides.

## Anomalous and spin Hall transport in a ferromagnet

In a conductor whose electron spins align perpendicular to the `xy` plane, spin-orbit skew scattering deflects spin-up and spin-down electrons sideways in `spin_plane` through opposite angles even at zero external magnetic field. Spin-up and spin-down electrons then form two carrier species sharing negative charge, negative entropy per charge, unit mobility, and one rapidly equilibrated electrochemical potential, each turned by `1 - skew * spin_plane`. Sweeping magnetization (spin-up share minus spin-down share) from `+1` through `0` to `-1` at zero field traces both magnetized and unmagnetized spin transport. Running and rendering the magnetization sweep takes about four seconds.

In [ ]:
spin_valences = np.array([-1, -1])                                            # [2]
spin_entropies_per_charge = np.array([-1.0, -1.0])                            # [2]
spin_mobilities = np.array([1.0, 1.0])                                        # [2]
spin_skews = np.array([1.0, -1.0])                                            # [2]
magnetizations = np.cos(np.linspace(0, np.pi, n_frames))                      # [n_frames]
spin_densities = np.stack([(1 + magnetizations) / 2, (1 - magnetizations) / 2], axis=-1)   # [n_frames, 2]
no_field = np.zeros(n_frames) * mv.xy                                         # [n_frames] Magnetic+

In [ ]:
spin_carriers = (spin_valences, spin_entropies_per_charge, spin_mobilities, spin_skews)
spin_frames = list(zip(
    no_field,
    steady_states(bar_topology, bar_areas, bar_gradients, left_contacts, right_contacts, current,
                  temperature_fall * scenarios.length, no_field, *spin_carriers, spin_densities, conductivity),
    steady_states(ring_topology, ring_areas, ring_gradients, inner_contacts, outer_contacts, current,
                  temperature_fall * (1 - scenarios.inner_radius), no_field, *spin_carriers, spin_densities,
                  conductivity)))
spin_bar_states = [bar_state for _, bar_state, _ in spin_frames]
show_animation(render.current_driven_animation(*conductors, frames[0], spin_frames), "magnetotransport_spin",
               frame_duration_ms, n_frames=n_frames)

White streaks now show spin-up electrons and black streaks show spin-down electrons while the zero-field indicator between the conductors remains a point. At full spin-up polarization, skew scattering tilts the bar's equipotentials in the anomalous Hall effect; in the unmagnetized midpoint, spin-up and spin-down electrons deflect to opposite edges so transverse charge currents cancel, equipotentials square up, longitudinal resistance rises, and a pure spin Hall current crosses the channel. Because both spin species carry identical entropy per charge, their transverse entropy currents cancel as well and the bar stays uniformly at equilibrium grey.

In [ ]:
spin_electric = stack([potentials for _, potentials, _, _, _ in spin_bar_states])    # [n_frames] Potential+[V]
spin_thermal = stack([potentials for _, _, _, potentials, _ in spin_bar_states])     # [n_frames] Potential+[V]
spin_voltages = (voltmeters * (spin_electric & time)).batch() / current              # [n_frames, 2] Scalar+
spin_thermal_voltages = (voltmeters * (spin_thermal & time)).batch()                 # [n_frames, 2] Scalar+
spin_temperatures = (voltmeters * (spin_thermal & entropy)).batch()                  # [n_frames, 2] Scalar+
spin_resistance, spin_hall_resistance = spin_voltages[:, 0], spin_voltages[:, 1]     # [n_frames] Scalar+, [n_frames] Scalar+
spin_nernst = -spin_thermal_voltages[:, 1] / spin_temperatures[:, 0]                 # [n_frames] Scalar+
render.spin_readings_figure(magnetizations, spin_resistance, spin_hall_resistance);

Anomalous Hall resistance tracks the magnetization from `+1` (all spin up) through zero when unmagnetized to `-1` (all spin down), while longitudinal resistance peaks at `1 + skew^2` in the unmagnetized state where no transverse Hall field counteracts the opposite spin deflections. Swapping spin populations reverses the magnetization and maps the transport operator to its `.adjugate()`, whereas under open-circuit heating both spins share one Seebeck coefficient and stand still, leaving anomalous and spin Nernst effects zero in energy-independent Drude scattering.

## Summary of transport effects

| Effect | Manifestation in the model |
| --- | --- |
| Hall | Tilted equipotentials and transverse Hall resistance in the current-driven single-carrier bar |
| Corbino | Spiralling radial currents and balance-invariant resistance `(1 + strength^2)` times the zero-field ring value |
| Nernst | Longitudinal equipotentials and peak transverse electric signal at compensation in the heat-driven bar |
| Ettingshausen | Edge warming at compensation in the current-driven bar as opposite charges deflect to the same side |
| Righi–Leduc | Transverse temperature gradient in the heat-driven bar at intermediate electron–hole balances |
| Seebeck | Open-circuit longitudinal voltage cancelling pure-carrier thermal drift |
| Peltier | Entropy carried per unit charge `entropy_per_charge` in `velocity.regressive(parcel)` |
| Kelvin | Endpoint Seebeck thermopower matching the Peltier entropy per unit charge by parcel construction |
| Lorentz | Combined electromagnetic and thermal force `(magnetic + drive).commutator(parcel + valence * velocity)` |
| Drude | Steady drift velocity inverted via the scaled spatial rotor `1 - valence * mobility * magnetic` |
| Onsager | Positive invariant dissipation `drive & flux` and self-adjugate zero-field transport operator |
| Casimir | Transport operator mapped to `.adjugate()` upon reversing magnetic field or magnetization |
| Anomalous Hall | Zero-field transverse Hall resistance proportional to magnetization under spin skew scattering |
| Spin Hall | Pure transverse spin separation and longitudinal resistance peak in the unmagnetized bar |
| Smit | Spin-dependent skew-scattering rotor `1 - skew * spin_plane` turning opposite spins opposite ways |
| Mott | Two-current parallel sum over spin-up and spin-down carrier populations |

In [ ]:
# checks
# Hall rotor angles equal -valence * arctan(mobility * strength), and compensated charge and entropy fluxes separate orthogonally.
expected_angles = -valences * np.arctan(mobilities * strength)
assert ((hall_angles - expected_angles).abs() < 1e-12).all()
compensated_flux = (end_fluxes[0] + end_fluxes[1]) / 2                        # [] Flux-
charge_along, charge_across = mv.xt & compensated_flux, mv.yt & compensated_flux     # [] Scalar+, [] Scalar+
entropy_along, entropy_across = mv.xs & compensated_flux, mv.ys & compensated_flux   # [] Scalar+, [] Scalar+
assert charge_across.abs() < 1e-12 * charge_along.abs() and entropy_along.abs() < 1e-12 * entropy_across.abs()

# Onsager-Casimir reciprocity: reversing the magnetic field maps the transport operator to its adjugate.
forward_responses = response(magnetic_sweep, *carriers, densities, conductivity)      # [n_frames] Flux- <- Drive+
reversed_responses = response(-magnetic_sweep, *carriers, densities, conductivity)    # [n_frames] Flux- <- Drive+
scale = np.abs(forward_responses.kernel).max()
np.testing.assert_allclose(reversed_responses.kernel, forward_responses.adjugate().kernel, atol=1e-12 * scale)

# Dissipation is strictly positive on every bar triangle under both drives, and the antisymmetric part dissipates zero.
bar_corners = bar_topology.incidence[2, 0]                                    # [3] Scalar+[F <- V]
bar_drives = -(bar_gradients ^ (bar_corners * stack((electric_potentials, thermal_potentials))[:, :, None])).sum(axis=-1)   # [2, n_frames] Drive+[F]
dissipation = bar_drives & forward_responses(bar_drives)                      # [2, n_frames] Scalar+[F]
assert (dissipation > 0).all()
odd_responses = (forward_responses - reversed_responses) / 2                  # [n_frames] Flux- <- Drive+
assert (bar_drives & odd_responses(bar_drives)).abs().max() < 1e-12 * dissipation.max()

# Commutator force equals spatial drag while magnetic power vanishes and total carrier power matches electric work.
middle = n_frames // 2
_, compensated_electric, compensated_velocities, _, _ = bar_states[middle]
compensated_drive = bar_drives[0, middle]                                     # Drive+[F]
compensated_bar_flux = forward_responses[middle](compensated_drive)           # Flux-[F]
moving_parcels = parcels + valences * compensated_velocities                  # [2] Point-[F]
pushes = (magnetic + compensated_drive).commutator(moving_parcels)            # [2] Point-[F]
through_space = stack((mv.x, mv.y))[:, None]                                  # [2, 1] Plane+
assert (((through_space & pushes) - (through_space & (compensated_velocities / mobilities))).abs() < 1e-10).all()
assert (mv.t & magnetic.commutator(moving_parcels)).abs().max() < 1e-12
electric_field = -(bar_gradients ^ ((bar_corners * (compensated_electric & time)) * mv.t)).sum(axis=0)   # Bivector+[F]
carrier_power = (densities[middle] * (mv.t & pushes)).sum(axis=0)             # Scalar+[F]
assert ((carrier_power - (electric_field & compensated_bar_flux)).abs() < 1e-10).all()

# Lorentz boost along x mixes magnetic strength into transverse electric field while preserving dissipation.
rapidity = 0.3
boost = (mv.xt * (rapidity / 2)).exp()                                        # [] Rotor+
spacetime_field = electric_field + magnetic                                   # Bivector+[F]
swept_across = y_direction & time                                             # [] Flux-
rest_reading = spacetime_field & swept_across                                 # Scalar+[F]
moving_reading = (boost >> spacetime_field) & swept_across                    # Scalar+[F]
lorentz_reading = np.cosh(rapidity) * rest_reading + np.sinh(rapidity) * strength
assert ((moving_reading - lorentz_reading).abs() < 1e-10).all()
assert ((((boost >> compensated_drive) & (boost >> compensated_bar_flux))
         - (compensated_drive & compensated_bar_flux)).abs() < 1e-10).all()

# Hall bar and Corbino ring endpoint, compensation, and circulation invariants.
assert ((resistance[[0, -1]] - 1.0).abs() < 1e-2).all()
assert ((hall_resistance[[0, -1]] - strength / valences).abs() < 2e-2 * strength).all()
assert ((seebeck[[0, -1]] - entropies_per_charge).abs() < 1e-10).all()
assert (nernst[[0, -1]].abs() < 1e-10).all()
assert hall_resistance[middle].abs() < 1e-12 * strength
assert nernst.abs().argmax() == middle
assert (righi_leduc[[0, middle, -1]].abs() < 1e-10).all()
assert righi_leduc[middle // 2] > 0 > righi_leduc[-1 - middle // 2]

ring_electric = stack([potentials for _, potentials, _, _, _ in ring_states])   # [n_frames] Potential+[V]
rim_voltmeter = ring_topology.reading(inner_contacts[:1], 0) - ring_topology.reading(outer_contacts[:1], 0)   # Scalar+[1 <- V]
ring_resistances = (rim_voltmeter * (ring_electric & time)).batch()[:, 0] / current   # [n_frames] Scalar+
corbino_resistance = (1 + strength ** 2) * np.log(1 / scenarios.inner_radius) / (2 * np.pi)
assert ((ring_resistances - ring_resistances[0]).abs() < 1e-10).all()
assert (ring_resistances[0] - corbino_resistance).abs() < 1e-2 * corbino_resistance

_, _, _, _, single_drifts = ring_states[0]
assert single_drifts[0].antinorm().max() < 1e-10
_, _, _, compensated_potentials, compensated_drifts = ring_states[middle]
assert (compensated_potentials & time).abs().max() < 1e-12
ring_corners = ring_topology.incidence[2, 0]                                  # [3] Scalar+[F <- V]
radial_offsets = ring_corners.mean(axis=0) * ring_vertices - origin           # Point-[F]
circulation = (ring_areas * (mv.xy & (radial_offsets & compensated_drifts))).sites.sum()   # [2] Scalar+
assert circulation[0] * circulation[1] < 0

# Ferromagnet anomalous Hall, spin Hall peak, zero Drude Nernst signal, and magnetization Onsager-Casimir symmetry.
assert ((spin_hall_resistance[[0, -1]] - spin_skews).abs() < 2e-2).all()
assert spin_hall_resistance[middle].abs() < 1e-12
assert spin_resistance.argmax() == middle
assert spin_nernst.abs().max() < 1e-10
spin_responses = response(no_field, *spin_carriers, spin_densities, conductivity)   # [n_frames] Flux- <- Drive+
spin_scale = np.abs(spin_responses.kernel).max()
np.testing.assert_allclose(spin_responses[::-1].kernel, spin_responses.adjugate().kernel, atol=1e-12 * spin_scale)